In [ ]:
from google.colab import drive
import os
import json
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "figures"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset shape:", feature_df.shape)
print("Stations:", feature_df["StationId"].nunique())

In [ ]:
dl_numeric_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Hour",
    "Day",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

dl_categorical_feature = "StationId"

target = "AQI_t_plus_1"

print("Deep-learning numeric features:", len(dl_numeric_features))
print("Target:", target)

In [ ]:
unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[split_position - 1]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("Train:")
print(train_df["Datetime"].min(), "→", train_df["Datetime"].max())

print("\nTest:")
print(test_df["Datetime"].min(), "→", test_df["Datetime"].max())

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

numeric_imputer = SimpleImputer(
    strategy="median"
)

numeric_scaler = StandardScaler()

X_train_numeric = numeric_imputer.fit_transform(
    train_df[dl_numeric_features]
)

X_train_numeric = numeric_scaler.fit_transform(
    X_train_numeric
)

X_test_numeric = numeric_imputer.transform(
    test_df[dl_numeric_features]
)

X_test_numeric = numeric_scaler.transform(
    X_test_numeric
)

station_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_station = station_encoder.fit_transform(
    train_df[[dl_categorical_feature]]
)

X_test_station = station_encoder.transform(
    test_df[[dl_categorical_feature]]
)

print("Numeric training matrix:", X_train_numeric.shape)
print("Station encoding:", X_train_station.shape)

In [ ]:
X_train_processed = np.hstack([
    X_train_numeric,
    X_train_station
]).astype("float32")

X_test_processed = np.hstack([
    X_test_numeric,
    X_test_station
]).astype("float32")

y_train_values = (
    train_df[target]
    .to_numpy()
    .astype("float32")
)

y_test_values = (
    test_df[target]
    .to_numpy()
    .astype("float32")
)

print("Processed train matrix:", X_train_processed.shape)
print("Processed test matrix :", X_test_processed.shape)

In [ ]:
train_processed = train_df[
    ["StationId", "Datetime"]
].copy()

test_processed = test_df[
    ["StationId", "Datetime"]
].copy()

train_processed["Target"] = y_train_values
test_processed["Target"] = y_test_values

print(train_processed.head())

In [ ]:
LOOKBACK = 12

def build_sequences(
    original_df,
    processed_features,
    target_values,
    lookback
):
    sequences = []
    targets = []
    metadata = []

    processed_df = original_df[
        ["StationId", "Datetime"]
    ].reset_index(drop=True)

    for station_id in processed_df["StationId"].unique():

        station_positions = np.where(
            processed_df["StationId"].to_numpy()
            == station_id
        )[0]

        station_times = (
            processed_df.iloc[station_positions]["Datetime"]
            .to_numpy()
        )

        station_features = processed_features[
            station_positions
        ]

        station_targets = target_values[
            station_positions
        ]

        for i in range(
            lookback - 1,
            len(station_positions)
        ):

            # Require the sequence itself to be hourly-continuous
            start_time = pd.Timestamp(
                station_times[i - lookback + 1]
            )

            end_time = pd.Timestamp(
                station_times[i]
            )

            expected_hours = (
                lookback - 1
            )

            actual_hours = (
                end_time - start_time
            ).total_seconds() / 3600

            if actual_hours != expected_hours:
                continue

            sequences.append(
                station_features[
                    i - lookback + 1:i + 1
                ]
            )

            targets.append(
                station_targets[i]
            )

            metadata.append(
                (
                    station_id,
                    station_times[i]
                )
            )

    return (
        np.asarray(sequences, dtype="float32"),
        np.asarray(targets, dtype="float32"),
        metadata
    )

print("Sequence builder ready.")

In [ ]:
X_train_seq, y_train_seq, train_metadata = build_sequences(
    train_df,
    X_train_processed,
    y_train_values,
    LOOKBACK
)

print("Training sequences:", X_train_seq.shape)
print("Training targets:", y_train_seq.shape)

In [ ]:
X_test_seq, y_test_seq, test_metadata = build_sequences(
    test_df,
    X_test_processed,
    y_test_values,
    LOOKBACK
)

print("Test sequences:", X_test_seq.shape)
print("Test targets:", y_test_seq.shape)

In [ ]:
# Convert sequence timestamps to a sorted NumPy array
sequence_times = np.sort(
    pd.to_datetime(
        [item[1] for item in train_metadata]
    ).to_numpy()
)

validation_cutoff_index = int(
    len(sequence_times) * 0.85
)

# Timestamp separating sequence-training and validation periods
train_sequence_end = sequence_times[
    validation_cutoff_index - 1
]

# Create masks
train_sequence_mask = (
    pd.to_datetime(
        [item[1] for item in train_metadata]
    ).to_numpy()
    <= train_sequence_end
)

val_sequence_mask = (
    pd.to_datetime(
        [item[1] for item in train_metadata]
    ).to_numpy()
    > train_sequence_end
)

# Split sequences
X_seq_train = X_train_seq[
    train_sequence_mask
]

y_seq_train = y_train_seq[
    train_sequence_mask
]

X_seq_val = X_train_seq[
    val_sequence_mask
]

y_seq_val = y_train_seq[
    val_sequence_mask
]

print("Sequence training shape:", X_seq_train.shape)
print("Sequence validation shape:", X_seq_val.shape)

print(
    "\nTraining sequence end:",
    train_sequence_end
)

print(
    "Validation sequence start:",
    sequence_times[
        validation_cutoff_index
    ]
)

print(
    "\nLeakage-free validation:",
    train_sequence_end
    < sequence_times[validation_cutoff_index]
)

In [ ]:
BATCH_SIZE = 256

train_ds = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_seq_train, y_seq_train)
    )
    .shuffle(
        buffer_size=min(
            len(X_seq_train),
            100000
        ),
        seed=42,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices(
        (X_seq_val, y_seq_val)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("TensorFlow datasets created.")

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

n_features = X_seq_train.shape[2]

lstm_model = Sequential([
    LSTM(
        64,
        input_shape=(
            LOOKBACK,
            n_features
        ),
        return_sequences=False
    ),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

lstm_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

print("Training callbacks configured.")

In [ ]:
lstm_history = lstm_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

print("LSTM training completed.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lstm_pred = (
    lstm_model
    .predict(
        X_test_seq,
        batch_size=BATCH_SIZE,
        verbose=1
    )
    .ravel()
)

lstm_mae = mean_absolute_error(
    y_test_seq,
    lstm_pred
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test_seq,
        lstm_pred
    )
)

lstm_r2 = r2_score(
    y_test_seq,
    lstm_pred
)

print("LSTM")
print("----")
print("MAE :", round(lstm_mae, 6))
print("RMSE:", round(lstm_rmse, 6))
print("R²  :", round(lstm_r2, 6))

In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional

cnn_bilstm_model = Sequential([
    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu",
        input_shape=(
            LOOKBACK,
            n_features
        )
    ),
    MaxPooling1D(
        pool_size=2
    ),
    Bidirectional(
        LSTM(
            64,
            return_sequences=False
        )
    ),
    Dropout(0.2),
    Dense(
        32,
        activation="relu"
    ),
    Dense(1)
])

cnn_bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

cnn_bilstm_model.summary()

In [ ]:
cnn_bilstm_history = cnn_bilstm_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

print("CNN-BiLSTM training completed.")

In [ ]:
# Train CNN-BiLSTM with more patience
cnn_bilstm_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

cnn_bilstm_history = cnn_bilstm_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=cnn_bilstm_callbacks,
    verbose=1
)

print(
    "CNN-BiLSTM epochs completed:",
    len(cnn_bilstm_history.history["loss"])
)

In [ ]:
cnn_bilstm_pred = (
    cnn_bilstm_model
    .predict(
        X_test_seq,
        batch_size=BATCH_SIZE,
        verbose=1
    )
    .ravel()
)

cnn_bilstm_mae = mean_absolute_error(
    y_test_seq,
    cnn_bilstm_pred
)

cnn_bilstm_rmse = np.sqrt(
    mean_squared_error(
        y_test_seq,
        cnn_bilstm_pred
    )
)

cnn_bilstm_r2 = r2_score(
    y_test_seq,
    cnn_bilstm_pred
)

print("CNN-BiLSTM")
print("----------")
print("MAE :", round(cnn_bilstm_mae, 6))
print("RMSE:", round(cnn_bilstm_rmse, 6))
print("R²  :", round(cnn_bilstm_r2, 6))

In [ ]:
classical_results = pd.read_csv(
    os.path.join(
        TABLE_PATH,
        "phase6_classical_model_results.csv"
    )
)

deep_learning_results = pd.DataFrame({
    "Model": [
        "LSTM",
        "CNN-BiLSTM"
    ],
    "MAE": [
        lstm_mae,
        cnn_bilstm_mae
    ],
    "RMSE": [
        lstm_rmse,
        cnn_bilstm_rmse
    ],
    "R2": [
        lstm_r2,
        cnn_bilstm_r2
    ]
})

combined_results = pd.concat(
    [
        classical_results[
            ["Model", "MAE", "RMSE", "R2"]
        ],
        deep_learning_results
    ],
    ignore_index=True
)

display(
    combined_results.sort_values(
        "RMSE"
    )
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    lstm_history.history["loss"],
    label="LSTM Train"
)

plt.plot(
    lstm_history.history["val_loss"],
    label="LSTM Validation"
)

plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "lstm_training_history.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    cnn_bilstm_history.history["loss"],
    label="CNN-BiLSTM Train"
)

plt.plot(
    cnn_bilstm_history.history["val_loss"],
    label="CNN-BiLSTM Validation"
)

plt.title("CNN-BiLSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "cnn_bilstm_training_history.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
lstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase8_lstm.keras"
    )
)

cnn_bilstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase8_cnn_bilstm.keras"
    )
)

print("Deep-learning models saved.")

In [ ]:
import joblib

joblib.dump(
    numeric_imputer,
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_imputer.joblib"
    )
)

joblib.dump(
    numeric_scaler,
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_scaler.joblib"
    )
)

joblib.dump(
    station_encoder,
    os.path.join(
        MODEL_PATH,
        "phase8_station_encoder.joblib"
    )

)

print("Deep-learning preprocessing objects saved.")

In [ ]:
deep_results_path = os.path.join(
    TABLE_PATH,
    "phase8_deep_learning_results.csv"
)

deep_learning_results.to_csv(
    deep_results_path,
    index=False
)

combined_results_path = os.path.join(
    TABLE_PATH,
    "phase8_all_model_results.csv"
)

combined_results.to_csv(
    combined_results_path,
    index=False
)

print("Phase 8 result tables saved.")

In [ ]:
phase8_summary = {
    "completed_phase": "Phase 8 - Deep Learning Models",
    "next_phase": "Phase 9 - VMD and PSO-VMD Experiments",

    "lookback_hours": LOOKBACK,

    "lstm": {
        "MAE": float(lstm_mae),
        "RMSE": float(lstm_rmse),
        "R2": float(lstm_r2)
    },

    "cnn_bilstm": {
        "MAE": float(cnn_bilstm_mae),
        "RMSE": float(cnn_bilstm_rmse),
        "R2": float(cnn_bilstm_r2)
    },

    "train_sequences": int(len(X_seq_train)),
    "validation_sequences": int(len(X_seq_val)),
    "test_sequences": int(len(X_test_seq))
}

phase8_summary_path = os.path.join(
    OUTPUT_PATH,
    "phase8_summary.json"
)

with open(
    phase8_summary_path,
    "w"
) as f:
    json.dump(
        phase8_summary,
        f,
        indent=4
    )

print("Phase 8 summary saved.")

In [ ]:
print("=" * 60)
print("PHASE 8 — DEEP LEARNING SUMMARY")
print("=" * 60)

display(
    deep_learning_results
)

print(
    "\nBest deep-learning model:",
    deep_learning_results
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

print("=" * 60)

In [ ]:
import os
import json
import joblib

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save deep-learning models
# --------------------------------------------------

lstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase8_lstm.keras"
    )
)

cnn_bilstm_model.save(
    os.path.join(
        MODEL_PATH,
        "phase8_cnn_bilstm.keras"
    )
)

# --------------------------------------------------
# 2. Save preprocessing objects
# --------------------------------------------------

joblib.dump(
    numeric_imputer,
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_imputer.joblib"
    )
)

joblib.dump(
    numeric_scaler,
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_scaler.joblib"
    )
)

joblib.dump(
    station_encoder,
    os.path.join(
        MODEL_PATH,
        "phase8_station_encoder.joblib"
    )
)

# --------------------------------------------------
# 3. Save deep-learning results
# --------------------------------------------------

deep_learning_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase8_deep_learning_results.csv"
    ),
    index=False
)

combined_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase8_all_model_results.csv"
    ),
    index=False
)

# --------------------------------------------------
# 4. Save Phase 8 summary
# --------------------------------------------------

phase8_summary = {
    "completed_phase": "Phase 8 - Deep Learning Models",
    "next_phase": "Phase 9 - VMD and PSO-VMD Experiments",

    "lookback_hours": int(LOOKBACK),

    "best_deep_learning_model": "LSTM",

    "lstm": {
        "MAE": float(lstm_mae),
        "RMSE": float(lstm_rmse),
        "R2": float(lstm_r2)
    },

    "cnn_bilstm": {
        "MAE": float(cnn_bilstm_mae),
        "RMSE": float(cnn_bilstm_rmse),
        "R2": float(cnn_bilstm_r2)
    },

    "train_sequences": int(len(X_seq_train)),
    "validation_sequences": int(len(X_seq_val)),
    "test_sequences": int(len(X_test_seq))
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase8_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        phase8_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 8 - Deep Learning Models",
    "next_phase": "Phase 9 - VMD and PSO-VMD Experiments",
    "best_deep_learning_model": "LSTM"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_path, "w") as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 6. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    os.path.join(
        MODEL_PATH,
        "phase8_lstm.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase8_cnn_bilstm.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_imputer.joblib"
    ),
    os.path.join(
        MODEL_PATH,
        "phase8_numeric_scaler.joblib"
    ),
    os.path.join(
        MODEL_PATH,
        "phase8_station_encoder.joblib"
    ),
    os.path.join(
        TABLE_PATH,
        "phase8_deep_learning_results.csv"
    ),
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 8 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 8")
print("Best deep-learning model: LSTM")
print("Next: Phase 9 - VMD and PSO-VMD Experiments")